# 🎤 super_Voz - StyleTTS2 One-Click Pipeline

Este notebook executa o treinamento completo do StyleTTS2 usando seus áudios do **Cloudflare R2**.

### 🚀 Como usar:
1. Vá em **Ambiente de execução** -> **Alterar tipo de ambiente de execução** e selecione **GPU**.
2. Abra o arquivo `styletts2_colab_config.yml` no menu lateral e insira suas credenciais do Cloudflare R2.
3. Clique em **Ambiente de execução** -> **Executar tudo** (ou pressione `Ctrl + F9`).

### Política do Resemble Enhance
O `resemble-enhance` fica **ativado por padrão no Colab** em modo GPU. A integração usa `device="cuda"` diretamente na inferência oficial da biblioteca e mantém o tensor de entrada em CPU para evitar mistura manual de devices.

Para desativar o enhancer e preservar os originais, defina `SUPER_VOZ_ENABLE_RESEMBLE=0` antes de iniciar o pipeline. Mesmo com GPU, toda saída do Resemble é validada antes de entrar no dataset.

**Nota:** Este notebook inclui um script Keep-Alive para evitar que o Colab desconecte por inatividade durante o treino.

In [ ]:
# @title 🚀 INICIAR TUDO (ONE-CLICK)
import IPython
from pathlib import Path
import os
import subprocess
import sys
import shutil

# Resemble Enhance fica ligado por padrão no Colab e usa GPU via device="cuda".
# Para desativar, mude para "0" antes de executar tudo.
os.environ.setdefault("SUPER_VOZ_ENABLE_RESEMBLE", "1")

# 1. Montar Google Drive para salvar progresso
from google.colab import drive
print("--- 1. Montando Google Drive ---")
drive.mount('/content/drive')

# 2. Prevenir desconexão por inatividade (Keep-Alive)
print("--- 2. Ativando Keep-Alive ---")
display(IPython.display.Javascript('''
 function ClickConnect(){
   console.log("Mantendo conexão ativa...");
   document.querySelector("colab-connect-button").click()
 }
 setInterval(ClickConnect, 60000)
'''))

def setup_environment():
    REPO_URL = "https://github.com/warllemedicao/Voz_styllett2.git"
    REPO_BASE_DIR = Path("/content/Voz_styllett2")

    print(f"--- 3. Clonando/Atualizando Repositório: {REPO_URL} ---")
    if REPO_BASE_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_BASE_DIR), "fetch", "--all"], check=False)
        subprocess.run(["git", "-C", str(REPO_BASE_DIR), "reset", "--hard", "origin/main"], check=False)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_BASE_DIR)], check=True)

    print("--- 4. Localizando diretório do projeto ---")
    target_script = "run_colab_styletts2.py"
    found_path = None
    for p in REPO_BASE_DIR.rglob(target_script):
        found_path = p.parent.parent
        break
    
    if not found_path:
        found_path = REPO_BASE_DIR

    PROJECT_DIR = found_path.resolve()
    os.chdir(PROJECT_DIR)
    
    print(f"--- 5. Instalando dependências críticas ---")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "boto3"], check=True)

    print("--- 6. Verificando Hardware e Aceleração ---")
    try:
        import torch
        if torch.cuda.is_available():
            print(f"✅ GPU Detectada: {torch.cuda.get_device_name(0)}")
            try:
                import onnxruntime as ort
                if 'CUDAExecutionProvider' not in ort.get_available_providers():
                    print("⚠️ onnxruntime-gpu mal configurado. Corrigindo...")
                    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "onnxruntime", "onnxruntime-gpu"], check=True)
                    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"], check=True)
                else:
                    print("✅ Motor GPU ONNX pronto.")
            except ImportError:
                print("ℹ️ Instalando onnxruntime-gpu...")
                subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxruntime-gpu"], check=True)
        else:
            print("❌ AVISO: GPU não detectada!")
    except Exception as e:
        print(f"ℹ️ Preparando ambiente: {e}")

    print("--- 7. Política do Resemble Enhance ---")
    if os.environ.get("SUPER_VOZ_ENABLE_RESEMBLE") == "1":
        print("✅ Resemble Enhance ativado em modo GPU. Saídas ruins serão rejeitadas pelo limpeza_ia.py.")
    else:
        print("ℹ️ Resemble Enhance desativado. O pipeline preservará originais e fará padronização final.")

    return PROJECT_DIR

PROJECT_DIR = setup_environment()
os.chdir(PROJECT_DIR)

cmd = [sys.executable, "-u", "scripts/run_colab_styletts2.py", "--config", "styletts2_colab_config.yml"]

print("="*60)
print("INICIANDO PIPELINE SUPER_VOZ")
print("="*60)
print(f"Comando: {' '.join(cmd)}")

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")

code = proc.wait()
if code != 0:
    print(f"\n[ERRO] O pipeline falhou com código {code}.")
else:
    print("\n✅ Pipeline finalizado com sucesso!")